# Phishing URL Classification — Ensemble Model

This notebook trains all base models and combines them into a **soft-voting ensemble**.

### Base models
| # | Model | Data track |
|---|---|---|
| 1 | Logistic Regression (no penalty) | Linear (scaled) |
| 2 | Logistic Regression L2 | Linear (scaled) |
| 3 | Logistic Regression L1 | Linear (scaled) |
| 4 | Elastic Net | Linear (scaled) |
| 5 | PCR (PCA + Logistic) | Linear (scaled) |
| 6 | Forward Stepwise Logit | Linear (scaled) |
| 7 | Backward Stepwise Logit | Linear (scaled) |
| 8 | Random Forest | Tree (unscaled) |
| 9 | XGBoost | Tree (unscaled) |

### Ensemble strategy
**Soft voting** — each base model produces a probability for class 1 (legitimate).
The ensemble prediction is the unweighted average of these probabilities.
The final label is 1 if the average probability ≥ 0.5.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import warnings
import itertools

from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_curve, auc, roc_auc_score
)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
SEED = 42

## 1. Load Data

In [ ]:
# Linear track (scaled) — used by logistic / stepwise / PCR models
df_train_lin = pd.read_csv('../data/processed/train_lin.csv')
df_test_lin  = pd.read_csv('../data/processed/test_lin.csv')

X_train_lin = df_train_lin.drop('label', axis=1)
y_train     = df_train_lin['label']
X_test_lin  = df_test_lin.drop('label', axis=1)
y_test      = df_test_lin['label']

# Tree track (unscaled) — used by RF and XGBoost
df_train_tree = pd.read_csv('../data/processed/train_tree.csv')
df_test_tree  = pd.read_csv('../data/processed/test_tree.csv')

X_train_tree = df_train_tree.drop('label', axis=1)
X_test_tree  = df_test_tree.drop('label', axis=1)

print(f'Linear track  — train: {X_train_lin.shape}, test: {X_test_lin.shape}')
print(f'Tree track    — train: {X_train_tree.shape}, test: {X_test_tree.shape}')
print(f'Class balance — train legitimate rate: {y_train.mean():.3f}')

## 2. Cross-Validation Setup

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

folds_lin  = []
folds_tree = []

for train_idx, val_idx in skf.split(X_train_lin, y_train):
    folds_lin.append((
        X_train_lin.iloc[train_idx],  y_train.iloc[train_idx],
        X_train_lin.iloc[val_idx],    y_train.iloc[val_idx]
    ))
    folds_tree.append((
        X_train_tree.iloc[train_idx], y_train.iloc[train_idx],
        X_train_tree.iloc[val_idx],   y_train.iloc[val_idx]
    ))

print(f'Prepared {len(folds_lin)} folds (linear + tree tracks).')

## 3. Train Base Models

Each model is fit on the full training set using the best hyperparameters found
via 5-fold CV in the individual model notebooks (or re-tuned here for completeness).

### 3.1 Logistic Regression (no penalty)

In [ ]:
lr_none = LogisticRegression(penalty=None, random_state=SEED, max_iter=1000)
lr_none.fit(X_train_lin, y_train)
prob_lr_none = lr_none.predict_proba(X_test_lin)[:, 1]
print(f'LR (no penalty) — AUC: {roc_auc_score(y_test, prob_lr_none):.4f}')

### 3.2 Logistic Regression L2 (Ridge) — CV over lambda

In [ ]:
l2_lambdas = [0.01, 0.1, 1, 10, 100]
l2_cv_scores = {}

for lam in l2_lambdas:
    fold_accs = []
    for X_tr, y_tr, X_val, y_val in folds_lin:
        m = LogisticRegression(penalty='l2', C=1/lam, random_state=SEED, max_iter=1000)
        m.fit(X_tr, y_tr)
        fold_accs.append(accuracy_score(y_val, m.predict(X_val)))
    l2_cv_scores[lam] = np.mean(fold_accs)

best_l2_lam = max(l2_cv_scores, key=l2_cv_scores.get)
print(f'Best L2 lambda: {best_l2_lam}  (CV acc={l2_cv_scores[best_l2_lam]:.4f})')

lr_l2 = LogisticRegression(penalty='l2', C=1/best_l2_lam, random_state=SEED, max_iter=1000)
lr_l2.fit(X_train_lin, y_train)
prob_lr_l2 = lr_l2.predict_proba(X_test_lin)[:, 1]
print(f'LR L2 — AUC: {roc_auc_score(y_test, prob_lr_l2):.4f}')

### 3.3 Logistic Regression L1 (Lasso) — CV over lambda

In [ ]:
l1_lambdas = [0.01, 0.1, 1, 10]
l1_cv_scores = {}

for lam in l1_lambdas:
    fold_accs = []
    for X_tr, y_tr, X_val, y_val in folds_lin:
        m = LogisticRegression(penalty='l1', solver='saga', C=1/lam,
                               max_iter=2000, tol=1e-3)
        m.fit(X_tr, y_tr)
        fold_accs.append(accuracy_score(y_val, m.predict(X_val)))
    l1_cv_scores[lam] = np.mean(fold_accs)

best_l1_lam = max(l1_cv_scores, key=l1_cv_scores.get)
print(f'Best L1 lambda: {best_l1_lam}  (CV acc={l1_cv_scores[best_l1_lam]:.4f})')

lr_l1 = LogisticRegression(penalty='l1', solver='saga', C=1/best_l1_lam,
                            random_state=SEED, max_iter=2000)
lr_l1.fit(X_train_lin, y_train)
prob_lr_l1 = lr_l1.predict_proba(X_test_lin)[:, 1]
print(f'LR L1 — AUC: {roc_auc_score(y_test, prob_lr_l1):.4f}')

### 3.4 Elastic Net — CV over lambda and l1_ratio

In [ ]:
en_lambdas   = [0.01, 0.1, 1, 10, 100]
en_l1_ratios = [0.1, 0.5, 0.9]
en_cv_scores = {}

for lam, l1r in itertools.product(en_lambdas, en_l1_ratios):
    fold_accs = []
    for X_tr, y_tr, X_val, y_val in folds_lin:
        m = LogisticRegression(penalty='elasticnet', solver='saga',
                               C=1/lam, l1_ratio=l1r,
                               random_state=SEED, max_iter=1000, tol=1e-3)
        m.fit(X_tr, y_tr)
        fold_accs.append(accuracy_score(y_val, m.predict(X_val)))
    en_cv_scores[(lam, l1r)] = np.mean(fold_accs)

best_en_lam, best_en_l1r = max(en_cv_scores, key=en_cv_scores.get)
print(f'Best EN lambda={best_en_lam}, l1_ratio={best_en_l1r}  '
      f'(CV acc={en_cv_scores[(best_en_lam, best_en_l1r)]:.4f})')

lr_en = LogisticRegression(penalty='elasticnet', solver='saga',
                            C=1/best_en_lam, l1_ratio=best_en_l1r,
                            random_state=SEED, max_iter=1000, tol=1e-3)
lr_en.fit(X_train_lin, y_train)
prob_lr_en = lr_en.predict_proba(X_test_lin)[:, 1]
print(f'Elastic Net — AUC: {roc_auc_score(y_test, prob_lr_en):.4f}')

### 3.5 Principal Component Regression (PCR)

In [ ]:
# Use 9 components (chosen in ML_models_linear.ipynb from scree plot — ~90% variance)
N_COMPONENTS = 9

pca = PCA(n_components=N_COMPONENTS, random_state=SEED)
X_train_pca = pca.fit_transform(X_train_lin)
X_test_pca  = pca.transform(X_test_lin)

lr_pcr = LogisticRegression(random_state=SEED, max_iter=1000)
lr_pcr.fit(X_train_pca, y_train)
prob_pcr = lr_pcr.predict_proba(X_test_pca)[:, 1]
print(f'PCR ({N_COMPONENTS} components, {pca.explained_variance_ratio_.sum():.3f} var explained) '
      f'— AUC: {roc_auc_score(y_test, prob_pcr):.4f}')

### 3.6 Forward Stepwise Logit

In [ ]:
def forward_stepwise_selection(X, y, p_enter=0.01, max_features=None, verbose=False):
    selected  = []
    remaining = list(X.columns)
    if max_features is None:
        max_features = len(remaining)
    for step in range(max_features):
        best_feature, best_p = None, np.inf
        for feat in remaining:
            X_trial = sm.add_constant(X[selected + [feat]], has_constant='add')
            try:
                model = sm.Logit(y, X_trial).fit(disp=0, method='lbfgs', maxiter=200)
                pval  = model.pvalues.get(feat, np.inf)
            except Exception:
                continue
            if pval < best_p:
                best_p, best_feature = pval, feat
        if best_feature is None or best_p >= p_enter:
            break
        selected.append(best_feature)
        remaining.remove(best_feature)
        if verbose:
            print(f'Step {step+1}: added {best_feature!r} (p={best_p:.2e})')
    return selected


selected_fwd = forward_stepwise_selection(X_train_lin, y_train, p_enter=0.01, verbose=True)
print(f'\nForward selection: {len(selected_fwd)} features selected')

X_train_fwd_const = sm.add_constant(X_train_lin[selected_fwd], has_constant='add')
model_fwd = sm.Logit(y_train, X_train_fwd_const).fit(disp=0, method='lbfgs', maxiter=300)

X_test_fwd_const = sm.add_constant(X_test_lin[selected_fwd], has_constant='add')
prob_fwd = model_fwd.predict(X_test_fwd_const).values
print(f'Forward Stepwise Logit — AUC: {roc_auc_score(y_test, prob_fwd):.4f}')

### 3.7 Backward Stepwise Logit

In [ ]:
def backward_stepwise_selection(X, y, p_remove=0.05, min_features=1, verbose=False):
    selected = list(X.columns)
    while len(selected) > min_features:
        X_curr = sm.add_constant(X[selected], has_constant='add')
        try:
            model = sm.Logit(y, X_curr).fit(disp=0, method='lbfgs', maxiter=300)
        except Exception:
            break
        pvals = model.pvalues.drop(labels='const', errors='ignore').dropna()
        if pvals.empty:
            break
        worst_feat, worst_p = pvals.idxmax(), float(pvals.max())
        if worst_p <= p_remove:
            break
        selected.remove(worst_feat)
        if verbose:
            print(f'Removed {worst_feat!r} (p={worst_p:.2e}), remaining={len(selected)}')
    return selected


selected_bwd = backward_stepwise_selection(X_train_lin, y_train, p_remove=0.05, verbose=True)
print(f'\nBackward selection: {len(selected_bwd)} features remaining')

X_train_bwd_const = sm.add_constant(X_train_lin[selected_bwd], has_constant='add')
model_bwd = sm.Logit(y_train, X_train_bwd_const).fit(disp=0, method='lbfgs', maxiter=300)

X_test_bwd_const = sm.add_constant(X_test_lin[selected_bwd], has_constant='add')
prob_bwd = model_bwd.predict(X_test_bwd_const).values
print(f'Backward Stepwise Logit — AUC: {roc_auc_score(y_test, prob_bwd):.4f}')

### 3.8 Random Forest — CV over hyperparameters

In [ ]:
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth':    [None, 10, 20],
    'max_features': ['sqrt', 'log2'],
}
rf_cv_scores = {}

for n_est, max_d, max_f in itertools.product(
        rf_param_grid['n_estimators'],
        rf_param_grid['max_depth'],
        rf_param_grid['max_features']):
    fold_accs = []
    for X_tr, y_tr, X_val, y_val in folds_tree:
        rf = RandomForestClassifier(n_estimators=n_est, max_depth=max_d,
                                    max_features=max_f, n_jobs=-1, random_state=SEED)
        rf.fit(X_tr, y_tr)
        fold_accs.append(accuracy_score(y_val, rf.predict(X_val)))
    rf_cv_scores[(n_est, max_d, max_f)] = np.mean(fold_accs)
    print(f'n_est={n_est}, max_depth={str(max_d):4s}, max_features={max_f} '
          f'→ CV acc={np.mean(fold_accs):.4f}')

best_rf = max(rf_cv_scores, key=rf_cv_scores.get)
print(f'\nBest RF: n_estimators={best_rf[0]}, max_depth={best_rf[1]}, '
      f'max_features={best_rf[2]}  (CV acc={rf_cv_scores[best_rf]:.4f})')

rf_model = RandomForestClassifier(n_estimators=best_rf[0], max_depth=best_rf[1],
                                   max_features=best_rf[2], n_jobs=-1, random_state=SEED)
rf_model.fit(X_train_tree, y_train)
prob_rf = rf_model.predict_proba(X_test_tree)[:, 1]
print(f'Random Forest — AUC: {roc_auc_score(y_test, prob_rf):.4f}')

### 3.9 XGBoost — CV over hyperparameters

In [ ]:
xgb_param_grid = {
    'n_estimators':  [100, 200],
    'max_depth':     [3, 5, 7],
    'learning_rate': [0.05, 0.1],
}
xgb_cv_scores = {}

for n_est, max_d, lr in itertools.product(
        xgb_param_grid['n_estimators'],
        xgb_param_grid['max_depth'],
        xgb_param_grid['learning_rate']):
    fold_accs = []
    for X_tr, y_tr, X_val, y_val in folds_tree:
        xgb = XGBClassifier(n_estimators=n_est, max_depth=max_d, learning_rate=lr,
                             eval_metric='logloss', n_jobs=-1,
                             random_state=SEED, verbosity=0)
        xgb.fit(X_tr, y_tr)
        fold_accs.append(accuracy_score(y_val, xgb.predict(X_val)))
    xgb_cv_scores[(n_est, max_d, lr)] = np.mean(fold_accs)
    print(f'n_est={n_est}, max_depth={max_d}, lr={lr:.2f} → CV acc={np.mean(fold_accs):.4f}')

best_xgb = max(xgb_cv_scores, key=xgb_cv_scores.get)
print(f'\nBest XGB: n_estimators={best_xgb[0]}, max_depth={best_xgb[1]}, '
      f'learning_rate={best_xgb[2]}  (CV acc={xgb_cv_scores[best_xgb]:.4f})')

xgb_model = XGBClassifier(n_estimators=best_xgb[0], max_depth=best_xgb[1],
                           learning_rate=best_xgb[2],
                           eval_metric='logloss', n_jobs=-1,
                           random_state=SEED, verbosity=0)
xgb_model.fit(X_train_tree, y_train)
prob_xgb = xgb_model.predict_proba(X_test_tree)[:, 1]
print(f'XGBoost — AUC: {roc_auc_score(y_test, prob_xgb):.4f}')

## 4. Soft-Voting Ensemble

In [ ]:
# Collect all base-model probability vectors
base_probs = {
    'LR (no penalty)':   prob_lr_none,
    'LR L2':             prob_lr_l2,
    'LR L1':             prob_lr_l1,
    'Elastic Net':       prob_lr_en,
    'PCR':               prob_pcr,
    'Forward Stepwise':  prob_fwd,
    'Backward Stepwise': prob_bwd,
    'Random Forest':     prob_rf,
    'XGBoost':           prob_xgb,
}

# Soft-voting: unweighted average of all predicted probabilities
prob_ensemble = np.mean(list(base_probs.values()), axis=0)
pred_ensemble = (prob_ensemble >= 0.5).astype(int)

print(f'Ensemble (soft voting) — AUC: {roc_auc_score(y_test, prob_ensemble):.4f}')

## 5. Model Comparison

In [ ]:
def metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'Accuracy':  accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall':    recall_score(y_true, y_pred, zero_division=0),
        'F1':        f1_score(y_true, y_pred, zero_division=0),
        'AUC-ROC':   roc_auc_score(y_true, y_prob),
    }

all_probs = {**base_probs, 'Ensemble (soft vote)': prob_ensemble}

results = pd.DataFrame(
    {name: metrics(y_test, prob) for name, prob in all_probs.items()}
).T.round(4)

results = results.sort_values('AUC-ROC', ascending=False)
print(results.to_string())

## 6. ROC Curves — All Models vs Ensemble

In [ ]:
colors = [
    'lightsteelblue', 'cornflowerblue', 'royalblue', 'navy',
    'plum', 'mediumpurple', 'darkorchid',
    'lightcoral', 'tomato'
]

fig, ax = plt.subplots(figsize=(8, 6))

for (name, prob), color in zip(base_probs.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=1.2, alpha=0.7,
            label=f'{name} (AUC={roc_auc:.4f})')

# Ensemble on top
fpr_ens, tpr_ens, _ = roc_curve(y_test, prob_ensemble)
auc_ens = auc(fpr_ens, tpr_ens)
ax.plot(fpr_ens, tpr_ens, color='black', lw=2.5,
        label=f'Ensemble (AUC={auc_ens:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Base Models vs Soft-Voting Ensemble')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

## 7. Ensemble Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, pred_ensemble)
print('Ensemble — Confusion Matrix:')
print(f'  TN={cm[0,0]:,}, FP={cm[0,1]:,}')
print(f'  FN={cm[1,0]:,}, TP={cm[1,1]:,}')

fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)
tick_labels = ['Phishing (0)', 'Legitimate (1)']
ax.set_xticks([0, 1]); ax.set_xticklabels(tick_labels, rotation=15)
ax.set_yticks([0, 1]); ax.set_yticklabels(tick_labels)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Ensemble — Confusion Matrix')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=11)
plt.tight_layout()
plt.show()